# NC-SSM + NC-SSM-Large: TASLP Final Experiments
**NC-SSM 7.4K | NC-SSM-Large 10.2K | SS v3 Wiener Gain**

---

## Execution Order

| Step | Description | Time |
|------|-------------|------|
| **0** | Setup (GPU + Drive) | 30s |
| **1** | Clone + Data + Checkpoints | ~3min |
| **2** | Train NC-SSM + NC-SSM-Large | ~20min |
| **3** | Full 7-SNR Noise Evaluation | ~15min |
| **4** | SS v2 vs SS v3 Comparison | ~30min |
| **5** | SS+Bypass Optimization Sweep | ~30min |
| **6** | SM-SSM + SAGN Training (ablation) | ~20min |
| **7** | Full 6-Model Comparison | ~20min |
| **8** | Sigma Gate Analysis | ~5min |
| **9** | **NASG Training + Eval** (low-SNR improvement) | ~25min |

## Step 0: Setup

In [ ]:
# GPU check
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

# Mount Drive (for checkpoint persistence)
from google.colab import drive
drive.mount('/content/drive')

## Step 1: Clone + Data + Recovery
**세션 끊어져도 Drive에서 체크포인트 자동 복원**
- 기존 `NanoMamba/checkpoints_full`의 baseline 체크포인트도 복원
- 누락 모델 자동 감지

In [ ]:
%%time
import os, shutil, torch

# ============================================================
# 1. Clone code
# ============================================================
!rm -rf /content/NC-SSM-TASLP
!git clone https://github.com/DrJinHoChoi/NC-SSM-TASLP.git /content/NC-SSM-TASLP
os.chdir('/content/NC-SSM-TASLP')

# ============================================================
# 2. Download GSC V2
# ============================================================
if not os.path.exists('./data/speech_commands_v0.02'):
    !mkdir -p data && cd data && \
     wget -q http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz && \
     mkdir -p speech_commands_v0.02 && \
     tar -xzf speech_commands_v0.02.tar.gz -C speech_commands_v0.02 && \
     rm speech_commands_v0.02.tar.gz
    print('GSC V2 downloaded')
else:
    print('GSC V2 already exists')

# ============================================================
# 3. Recovery: Restore checkpoints from Drive
# ============================================================
DRIVE_NEW = '/content/drive/MyDrive/NC-SSM-TASLP/checkpoints_full'
DRIVE_OLD = '/content/drive/MyDrive/NanoMamba/checkpoints_full'
LOCAL_CKPT = './checkpoints_full'
os.makedirs(LOCAL_CKPT, exist_ok=True)

# Required models for full experiment
REQUIRED = {
    'NanoMamba-NC-Large':  'NC-SSM-Large (10.2K) — 최종 제안 모델',
    'NanoMamba-NC-Matched': 'NC-SSM (7.4K) — 기본 제안 모델',
    'BC-ResNet-1':          'BC-ResNet-1 (7.5K) — CNN baseline',
    'NanoMamba-Matched-DualPCEN-v2-SSMv2': 'NM-Matched (7.4K) — SSM baseline',
    'DS-CNN-S':             'DS-CNN-S (23.8K) — Large CNN',
    'NanoMamba-Matched-SM': 'SM-SSM (7.4K) — ablation',
    'SAGN':                 'SAGN (7.2K) — CNN+LSG ablation',
}

def restore_checkpoint(model_name):
    """Try to restore checkpoint from Drive (new path first, then old path)."""
    local_dir = os.path.join(LOCAL_CKPT, model_name)
    local_best = os.path.join(local_dir, 'best.pt')
    if os.path.exists(local_best):
        return True  # Already local

    # Try new Drive path first
    for drive_path in [DRIVE_NEW, DRIVE_OLD]:
        src = os.path.join(drive_path, model_name, 'best.pt')
        if os.path.exists(src):
            os.makedirs(local_dir, exist_ok=True)
            shutil.copy2(src, local_best)
            return True
    return False

print('='*60)
print('CHECKPOINT RECOVERY')
print('='*60)
found, missing = [], []
for name, desc in REQUIRED.items():
    if restore_checkpoint(name):
        ckpt = torch.load(os.path.join(LOCAL_CKPT, name, 'best.pt'),
                          map_location='cpu')
        acc = ckpt.get('val_acc', 'N/A')
        print(f'  ✓ {name}: val_acc={acc}')
        found.append(name)
    else:
        print(f'  ✗ {name}: NOT FOUND — needs training')
        missing.append(name)

print(f'\n  Recovered: {len(found)}/{len(REQUIRED)}')
if missing:
    print(f'  Missing:   {", ".join(missing)}')
    print(f'  → Run training steps for missing models')
else:
    print(f'  All checkpoints ready! Skip to Step 3 (eval)')

# ============================================================
# 4. Helper: save_to_drive() — call after any training
# ============================================================
def save_to_drive():
    """Sync all local checkpoints to Drive for persistence."""
    os.makedirs(DRIVE_NEW, exist_ok=True)
    for d in os.listdir(LOCAL_CKPT):
        src = os.path.join(LOCAL_CKPT, d, 'best.pt')
        if os.path.exists(src):
            dst_dir = os.path.join(DRIVE_NEW, d)
            os.makedirs(dst_dir, exist_ok=True)
            shutil.copy2(src, os.path.join(dst_dir, 'best.pt'))
    print(f'✓ Checkpoints synced to Drive ({DRIVE_NEW})')

## Step 2: Train NC-SSM + NC-SSM-Large

### 2a: NC-SSM (7,443 params)
- `d_model=20, d_state=6, n_layers=2`
- Target: ≥95.3% clean accuracy

### 2b: NC-SSM-Large (~10,191 params)
- `d_model=24, d_state=8, n_layers=2`
- `use_tiny_conv` 제거됨 (per-sub-band SNR estimation 교란 방지)
- Target: ≥95% clean accuracy

In [ ]:
import os, shutil

# ============================================================
# 2a: Train NC-SSM (7,443 params)
# ============================================================
print('='*60)
print('2a: Training NC-SSM (d_model=20, d_state=6, 7,443 params)')
print('='*60)
!python train_colab.py \
    --models NanoMamba-NC-Matched \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --epochs 30 \
    --noise_aug --noise_curriculum_v2 \
    --calibrate

save_to_drive()
print('\n✓ NC-SSM saved to Drive')

# ============================================================
# 2b: Train NC-SSM-Large (~10,191 params, TinyConv 제거)
# ============================================================
# 기존 NC-SSM-Large 체크포인트 삭제 (구조 변경됨)
old_ckpt = './checkpoints_full/NanoMamba-NC-Large'
if os.path.exists(old_ckpt):
    shutil.rmtree(old_ckpt)
    print(f'✗ Deleted old NC-SSM-Large checkpoint')
drive_ckpt = '/content/drive/MyDrive/NC-SSM-TASLP/checkpoints_full/NanoMamba-NC-Large'
if os.path.exists(drive_ckpt):
    shutil.rmtree(drive_ckpt)

print('\n' + '='*60)
print('2b: Training NC-SSM-Large (d_model=24, d_state=8, ~10,191 params)')
print('='*60)
!python train_colab.py \
    --models NanoMamba-NC-Large \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --epochs 30 \
    --noise_aug --noise_curriculum_v2 \
    --calibrate

save_to_drive()
print('\n✓ NC-SSM-Large saved to Drive')
print('\n' + '='*60)
print('BOTH MODELS TRAINED AND SAVED TO DRIVE')
print('='*60)

## Step 3: Full 7-SNR Noise Evaluation
NC-SSM-Large vs NC-SSM vs BC-ResNet-1

In [ ]:
# Full noise evaluation: 5 noise x 7 SNR = 35 conditions
!python train_colab.py \
    --models NanoMamba-NC-Large,NanoMamba-NC-Matched,BC-ResNet-1 \
    --eval_only \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --noise_types factory,white,babble,street,pink \
    --snr_range=-15,-10,-5,0,5,10,15 \
    --calibrate --calibrate_continuous

## Step 4: SS+Bypass Evaluation (v2 + v3 비교)
- **SS v2**: Subtractive, SF-weighted, oversubtract [0.5, 3.0]
- **SS v3** (NEW): Wiener gain, aggressive, oversubtract [1.0, 5.0], no SF weight

In [ ]:
# ============================================================
# SS v2 (기존): NC-SSM + NC-SSM-Large + BC-ResNet-1
# ============================================================
print('='*70)
print('SS v2: Subtractive + SF-weighted + Bypass v2')
print('='*70)
!python train_colab.py \
    --models NanoMamba-NC-Large,NanoMamba-NC-Matched,BC-ResNet-1 \
    --eval_only \
    --use_enhancer --enhancer_bypass \
    --ss_version v2 --bypass_version v2 \
    --bypass_threshold -2.0 --bypass_scale 3.0 --sf_range 2.0 \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --noise_types factory,white,babble,street,pink \
    --snr_range=-15,-10,-5,0,5,10,15 \
    --calibrate --calibrate_continuous

# ============================================================
# SS v3 (NEW): NC-SSM + NC-SSM-Large + BC-ResNet-1
# Wiener gain + aggressive alpha + no SF weight
# ============================================================
print('\n' + '='*70)
print('SS v3: Wiener gain + aggressive (alpha 1.0-5.0) + low floor')
print('='*70)
!python train_colab.py \
    --models NanoMamba-NC-Large,NanoMamba-NC-Matched,BC-ResNet-1 \
    --eval_only \
    --use_enhancer --enhancer_bypass \
    --ss_version v3 --bypass_version v2 \
    --bypass_threshold -2.0 --bypass_scale 3.0 --sf_range 2.0 \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --noise_types factory,white,babble,street,pink \
    --snr_range=-15,-10,-5,0,5,10,15 \
    --calibrate --calibrate_continuous

print('\n' + '='*70)
print('COMPARE: SS v2 vs v3 at -15dB (improvement) and 0dB (degradation)')
print('='*70)

## Step 5: SS+Bypass Optimization Sweep
**v4 파라미터가 최적인지 확인**: sf_range, bypass_threshold, bypass_scale

In [ ]:
# ============================================================
# SS+Bypass Parameter Sweep: Is v4 (sf_range=2.0) optimal?
# Test 4 configurations and compare at -15dB and 0dB
# ============================================================

configs = [
    # (name, sf_range, threshold, scale)
    ('v4-current',  2.0, -2.0, 3.0),  # Current default
    ('v4-tight',    1.0, -1.0, 4.0),  # Tighter bypass: more aggressive at 0dB
    ('v4-wide',     3.0, -3.0, 2.0),  # Wider range: more SS at moderate SNR
    ('v4-noadapt',  0.0, -2.0, 3.0),  # No SF adaptation: same threshold all noises
]

print('='*70)
print('SS+BYPASS v4 PARAMETER SWEEP')
print('='*70)

for name, sf_range, threshold, scale in configs:
    print(f'\n{"="*70}')
    print(f'Config: {name} (sf_range={sf_range}, threshold={threshold}, scale={scale})')
    print(f'{"="*70}')
    !python train_colab.py \
        --models NanoMamba-NC-Large \
        --eval_only \
        --use_enhancer --enhancer_bypass \
        --ss_version v2 --bypass_version v2 \
        --bypass_threshold {threshold} --bypass_scale {scale} --sf_range {sf_range} \
        --data_dir ./data \
        --checkpoint_dir ./checkpoints_full \
        --noise_types factory,white,babble,street,pink \
        --snr_range=-15,0 \
        --calibrate --calibrate_continuous

print('\n' + '='*70)
print('SWEEP COMPLETE: Compare -15dB improvement vs 0dB degradation')
print('Optimal config should maximize -15dB gain while minimizing 0dB loss')
print('='*70)

## Step 6: SM-SSM + SAGN Training (TASLP ablation baselines)
논문 스토리: NM-Matched → SM-SSM → NC-SSM 진화

In [ ]:
# Train SM-SSM (intermediate step: +38 params for sigma gate)
!python train_colab.py \
    --models NanoMamba-Matched-SM \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --epochs 30 \
    --noise_aug --noise_curriculum_v2 \
    --calibrate

save_to_drive()  # Recovery point after SM-SSM

# Train SAGN (CNN+LSG baseline: proves LSG alone on CNN < NC-SSM)
!python train_colab.py \
    --models SAGN \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --epochs 30 \
    --noise_aug --noise_curriculum_v2 \
    --calibrate

save_to_drive()  # Recovery point after SAGN

## Step 7: Full 6-Model TASLP Comparison
**All models × 5 noise × 7 SNR = complete Table V**

In [ ]:
# TASLP Table V: Complete 6-model noise evaluation
!python train_colab.py \
    --models NanoMamba-NC-Large,NanoMamba-NC-Matched,NanoMamba-Matched-SM,SAGN,BC-ResNet-1,NanoMamba-Matched-DualPCEN-v2-SSMv2,DS-CNN-S \
    --eval_only \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --noise_types factory,white,babble,street,pink \
    --snr_range=-15,-10,-5,0,5,10,15 \
    --calibrate --calibrate_continuous

# TASLP Table VII: SS+Bypass for all models
!python train_colab.py \
    --models NanoMamba-NC-Large,NanoMamba-NC-Matched,NanoMamba-Matched-SM,NanoMamba-Matched-DualPCEN-v2-SSMv2,BC-ResNet-1,DS-CNN-S,SAGN \
    --eval_only \
    --use_enhancer --enhancer_bypass \
    --ss_version v2 --bypass_version v2 \
    --bypass_threshold -2.0 --bypass_scale 3.0 --sf_range 2.0 \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --noise_types factory,white,babble,street,pink \
    --snr_range=-15,-10,-5,0,5,10,15 \
    --calibrate --calibrate_continuous

print('\n' + '='*60)
print('TASLP FULL COMPARISON COMPLETE')
print('='*60)

## Step 8: Sigma Gate Analysis (Fig. 3 data)
Per-sub-band selectivity visualization data

In [ ]:
# Sigma gate per-sub-band analysis for NC-SSM-Large
import torch, sys, os
sys.path.insert(0, '/content/NC-SSM-TASLP')
from train_colab import (
    SpeechCommandsDataset, analyze_selectivity_gates,
    generate_noise_signal, mix_audio_at_snr
)
from nanomamba import create_nanomamba_nc_large

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load NC-SSM-Large checkpoint
model = create_nanomamba_nc_large(n_classes=12).to(device)
ckpt_path = './checkpoints_full/NanoMamba-NC-Large/best.pt'
if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f'Loaded NC-SSM-Large: val_acc={ckpt.get("val_acc", "N/A")}')
else:
    print(f'Checkpoint not found: {ckpt_path}')
    print('Run Step 2 first!')

model.eval()

# Load dataset for analysis
val_dataset = SpeechCommandsDataset('./data/speech_commands_v0.02', split='validation')

# Analyze sigma gates
analyze_selectivity_gates(model, val_dataset, device=device)

## Step 9: NASG Training — Noise-Aware Selective Gating
**저 SNR에서 CNN upper bound를 넘는 구조**

| Component | Description | Params |
|-----------|-------------|--------|
| `nasg_scale` | SNR-dependent gate steepness (init=20) | 1/block |
| `nasg_bias` | Gate threshold (init=-5) | 1/block |
| **Total** | 2 blocks × 2 params | **+4** |

**이론적 원리:**
- Low SNR: `nasg_w → 0` → selective params → 0 → LTI mode → O(σ²)
- High SNR: `nasg_w → 1` → full selectivity preserved
- -15dB에서 effective selectivity: σ·g ≈ 0.001 (2,500× noise reduction)

### 9a: Train NC-SSM-Large-NASG (10,195 params)
### 9b: Compare with baseline at -15dB (target: White 34.1% → 50%+)

In [ ]:
import os, shutil

# ============================================================
# 9a: Train NC-SSM-Large-NASG (10,195 params)
#     Noise-Aware Selective Gating for low-SNR improvement
# ============================================================
print('='*70)
print('9a: Training NC-SSM-Large-NASG (d_model=24, d_state=8, +NASG)')
print('    → nasg_scale=20, nasg_bias=-5 (steep gate for noise suppression)')
print('='*70)

!python train_colab.py \
    --models NanoMamba-NC-Large-NASG \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --epochs 30 \
    --noise_aug --noise_curriculum_v2 --low_snr_boost \
    --calibrate

save_to_drive()
print('\n✓ NC-SSM-Large-NASG saved to Drive')

# ============================================================
# 9b: Full noise evaluation — NASG vs Baseline vs CNN
# ============================================================
print('\n' + '='*70)
print('9b: NASG vs Baseline vs CNN — Full noise evaluation')
print('='*70)

!python train_colab.py \
    --models NanoMamba-NC-Large-NASG,NanoMamba-NC-Large,BC-ResNet-1 \
    --eval_only \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --noise_types factory,white,babble,street,pink \
    --snr_range=-15,-10,-5,0,5,10,15 \
    --calibrate --calibrate_continuous

print('\n' + '='*70)
print('KEY METRICS TO CHECK:')
print('  White  -15dB: Baseline 34.1% → NASG target 50%+')
print('  White  -10dB: Baseline 59.8% → NASG target 65%+')
print('  Clean       : Must stay ≥95.0% (no regression)')
print('  Factory -15dB: Baseline 61.3% → Check improvement')
print('='*70)

## Step 9c: NASG + SS+Bypass Evaluation
**NASG 체크포인트에 SS v2/v3 + Bypass 적용하여 White/Pink 추가 개선 확인**

이론적 배경:
- SS가 input noise를 α배 줄이면 → SSM internal variance α⁶ 감소 (CNN은 α²만)
- NASG가 이미 selectivity를 suppression → SS와 시너지 기대
- White noise at -15dB: 41.6% → SS로 50%+ 목표

In [ ]:
# ============================================================
# 9c: NASG + SS+Bypass (v2 & v3) — White/Pink noise target
# ============================================================

# --- SS v2 + Bypass ---
print('='*70)
print('9c-1: NASG + SS v2 + Bypass')
print('='*70)
!python train_colab.py \
    --models NanoMamba-NC-Large-NASG \
    --eval_only \
    --use_enhancer --enhancer_bypass \
    --ss_version v2 --bypass_version v2 \
    --bypass_threshold -2.0 --bypass_scale 3.0 --sf_range 2.0 \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --noise_types factory,white,babble,street,pink \
    --snr_range=-15,-10,-5,0,5,10,15

# --- SS v3 (Wiener) + Bypass ---
print('\n' + '='*70)
print('9c-2: NASG + SS v3 (Wiener) + Bypass')
print('='*70)
!python train_colab.py \
    --models NanoMamba-NC-Large-NASG \
    --eval_only \
    --use_enhancer --enhancer_bypass \
    --ss_version v3 --bypass_version v2 \
    --bypass_threshold -2.0 --bypass_scale 3.0 --sf_range 2.0 \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --noise_types factory,white,babble,street,pink \
    --snr_range=-15,-10,-5,0,5,10,15

print('\n' + '='*70)
print('NASG + SS COMPARISON:')
print('  NASG only    — White -15dB: 41.6%')
print('  NASG + SS v2 — White -15dB: ???')
print('  NASG + SS v3 — White -15dB: ???')
print('  BC-ResNet-1  — White -15dB: 58.6% (target)')
print('='*70)

## Step 10: NASG v2 — Full White/Pink Noise Improvement Stack
**Phase 1-4 전체 적용 (새 코드 필요 → git pull 먼저)**

| Phase | Description | Params |
|-------|-------------|--------|
| 1A | White/Pink 가중 샘플링 (2×, SNR -3dB shift) | 0 |
| 1B | Per-sample loss weighting (extreme broadband 2×) | 0 |
| 2A | State update x temporal smoothing (broadband-gated) | 0 |
| 3A | Adaptive state masking (d_state 8→~5 at -15dB) | +4 |
| 4A | SS broadband oversubtraction α boost (3→4.5) | 0 |
| **Total** | NASG 4 + state_mask 4 | **10,199** |

**기존 NASG checkpoint는 state_mask 없어서 재학습 필요**

In [ ]:
import os, shutil

# ============================================================
# 10a: Update code (git pull for Phase 1-4 changes)
# ============================================================
print('='*70)
print('10a: Pulling latest code (Phase 1-4 improvements)')
print('='*70)
!cd /content/NC-SSM-TASLP && git pull origin main

# Delete old NASG checkpoint (incompatible: missing state_mask params)
old_nasg = './checkpoints_full/NanoMamba-NC-Large-NASG'
if os.path.exists(old_nasg):
    shutil.rmtree(old_nasg)
    print('✗ Deleted old NASG checkpoint (state_mask params added)')
drive_nasg = '/content/drive/MyDrive/NC-SSM-TASLP/checkpoints_full/NanoMamba-NC-Large-NASG'
if os.path.exists(drive_nasg):
    shutil.rmtree(drive_nasg)

# ============================================================
# 10b: Train NASG v2 (10,199 params)
#      Includes: state_mask + temporal smoothing + white/pink boost
# ============================================================
print('\n' + '='*70)
print('10b: Training NASG v2 (Phase 1-4 full stack)')
print('    New: state_mask (4p), temporal smoothing, white/pink boost')
print('    Total: 10,199 params')
print('='*70)

!python train_colab.py \
    --models NanoMamba-NC-Large-NASG \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --epochs 30 \
    --noise_aug --noise_curriculum_v2 --low_snr_boost \
    --calibrate

save_to_drive()
print('\n✓ NASG v2 saved to Drive')

# ============================================================
# 10c: Full eval — NASG v2 vs NASG v1 vs Baseline vs CNN
# ============================================================
print('\n' + '='*70)
print('10c: NASG v2 full noise evaluation')
print('='*70)

!python train_colab.py \
    --models NanoMamba-NC-Large-NASG \
    --eval_only \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --noise_types factory,white,babble,street,pink \
    --snr_range=-15,-10,-5,0,5,10,15

# ============================================================
# 10d: NASG v2 + SS v3 (broadband-boosted oversubtraction)
# ============================================================
print('\n' + '='*70)
print('10d: NASG v2 + SS v3 (broadband-boosted α)')
print('='*70)

!python train_colab.py \
    --models NanoMamba-NC-Large-NASG \
    --eval_only \
    --use_enhancer --enhancer_bypass \
    --ss_version v3 --bypass_version v2 \
    --bypass_threshold -2.0 --bypass_scale 3.0 --sf_range 2.0 \
    --data_dir ./data \
    --checkpoint_dir ./checkpoints_full \
    --noise_types factory,white,babble,street,pink \
    --snr_range=-15,-10,-5,0,5,10,15

print('\n' + '='*70)
print('NASG v2 RESULTS COMPARISON:')
print('  Baseline (no NASG)  — White -15dB: 34.1%')
print('  NASG v1 (Sep 9)     — White -15dB: 41.6%')
print('  NASG v2 (Step 10)   — White -15dB: ???  (target: 50%+)')
print('  NASG v2 + SS v3     — White -15dB: ???  (target: 55%+)')
print('  BC-ResNet-1         — White -15dB: 58.6%')
print('='*70)